# Survey frame review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** B3 — dark-majority follow-up: survey of "disappeared" firms  
(T11) ([docs/research-questions.md](../../docs/research-questions.md))
**Canonical computation:** `scripts/data/nano_survey_frame.py` (seed 20260712)  
**Data as of:** the liveness snapshot the frame was drawn from  

Companion view over the stratified survey frame, focused on stratification balance,
sampling diagnostics, and seed sensitivity. Exploratory-tier and non-citable; the
canonical frame is the committed CSV — do not redraw it here for fieldwork.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260806

## Data contract

- **Population:** the FIRM_ACTIVITY_ABSENT bucket of the dark-firm liveness file.
- **Grain:** firm.
- **Strata:** S1_active_evidence (high-confidence post-award patent activity),
  S2_holder_only (patents but no high-confidence post-award signal), S3_dark_core
  (no patent match). Allocation 20/20/35 with 2 ranked backups per primary.
- **Keys:** `company` within the frame; the liveness file's `normalized_name` upstream.
- **Determinism:** the canonical draw is fixed-seed (20260712). Any redraw in this
  notebook is diagnostic only and must use a declared seed.

In [ ]:
ARTIFACTS = {
    "survey frame": REPORT_DIR / "survey_frame.csv",
    "dark-firm liveness": REPORT_DIR / "dark_firm_liveness.csv",
}
GENERATORS = {
    "survey frame": "nano_survey_frame.py",
    "dark-firm liveness": "nano_dark_firm_liveness.py",
}
pd.DataFrame(
    [
        {"artifact": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)

def load_artifact(name: str) -> pd.DataFrame:
    """Read a canonical CSV artifact, or return an empty frame with a hint."""
    path = ARTIFACTS[name]
    if not path.exists():
        print(
            f"Missing {path.relative_to(REPO_ROOT)} — artifact not present; "
            f"run {GENERATORS[name]} first."
        )
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

## Allocation against population

Primary and backup counts per stratum, next to the population each stratum samples
from. Undersized strata (population smaller than allocation) surface here.

In [ ]:
frame = load_artifact("survey frame")
if frame.empty:
    allocation = pd.DataFrame()
else:
    allocation = (
        frame.pivot_table(index="stratum", columns="role", values="company", aggfunc="count")
        .fillna(0)
        .astype(int)
    )
allocation

## Stratification balance

Compare sampled primaries to their stratum population on the covariates the frame
carries (first award year, award count, patent count). Large gaps mean the sample
would misrepresent its stratum even before nonresponse.

In [ ]:
liveness = load_artifact("dark-firm liveness")
if frame.empty:
    balance = pd.DataFrame()
else:
    primaries = frame[frame["role"].eq("primary")].copy()
    for column in ("first_award_year", "awards_n", "any_patents_n"):
        primaries[column] = pd.to_numeric(primaries[column], errors="coerce")
    balance = primaries.groupby("stratum")[["first_award_year", "awards_n", "any_patents_n"]].median()
    if not liveness.empty:
        population = liveness[liveness["bucket"].eq("FIRM_ACTIVITY_ABSENT")].copy()
        for column in ("first_award_year", "awards_n", "any_patents_n"):
            if column in population.columns:
                population[column] = pd.to_numeric(population[column], errors="coerce")
        print("Population medians (FIRM_ACTIVITY_ABSENT) for comparison:")
        display(
            population[[c for c in ("first_award_year", "awards_n", "any_patents_n")
                        if c in population.columns]].median().rename("population_median").to_frame()
        )
balance

## Seed sensitivity (diagnostic only)

Redraw the S3 stratum under several declared seeds and measure overlap with the
canonical draw. Low overlap is expected for a small sample from a large stratum; what
matters is that stratum-level summary statistics stay stable across seeds.

In [ ]:
import random

if frame.empty or liveness.empty:
    seed_check = pd.DataFrame()
else:
    population = liveness[liveness["bucket"].eq("FIRM_ACTIVITY_ABSENT")]
    name_column = "normalized_name" if "normalized_name" in population.columns else "company"
    s3_pool = sorted(
        population.loc[
            pd.to_numeric(population.get("any_patents_n"), errors="coerce").fillna(0).eq(0),
            name_column,
        ].astype(str)
    )
    canonical = set(
        frame.loc[frame["stratum"].eq("S3_dark_core") & frame["role"].eq("primary"), "company"]
        .astype(str).str.upper()
    )
    rows = []
    for seed in (20260712, 1, 2, 3):
        rng = random.Random(seed)
        pool = list(s3_pool)
        rng.shuffle(pool)
        draw = {name.upper() for name in pool[: len(canonical)]}
        rows.append({
            "seed": seed,
            "drawn": len(draw),
            "overlap_with_canonical": len(draw & canonical),
        })
    seed_check = pd.DataFrame(rows)
    print("Note: overlap with the canonical frame is approximate — the canonical draw")
    print("shuffles display-name pools inside the script; treat this as a stability")
    print("diagnostic, not a reproduction.")
seed_check

## Fieldwork log

| Stratum | Primary reached? | Replacement used | Response state | Note |
|---|---|---|---|---|
| _Draft_ | _Yes/no_ | _Backup rank_ | _Complete/refused/unreachable_ | _Keep states separate_ |

Any change to allocation or strata definitions is a script change plus a redraw, with
the new seed recorded in the artifact — never an in-notebook edit.